# 00 · Setup: data download and dataset build

Downloads the Beijing dataset, verifies its SHA-256 and builds the cleaned dataset (run `pip install -e .[ml]` once beforehand). With `AIRQ_FIXTURE=1` a small synthetic dataset is used instead (CI, no internet).

*Estimated time: 2-5 min (download ~8 MB).*

In [ ]:
# --- parameters -------------------------------------------------------------------------
QUICK = True        # 2 stations, short run (< 5 min); set False for the full 12-station run
USE_DRIVE = False   # Colab only: mount Google Drive to keep results between sessions
RUN_OPTUNA = False  # hyper-parameter search (notebook 02 onwards)
SEEDS = [0]


In [ ]:
# --- setup (runs on Colab, skipped locally) ---------------------------------------------
import os
import subprocess
import sys
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/furkantopcuu/airq-iot.git"
if IN_COLAB:
    if not os.path.exists("/content/airq-iot"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/airq-iot"], check=True)
    os.chdir("/content/airq-iot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
    sys.path.insert(0, "/content/airq-iot/src")  # no restart needed after the editable install


In [ ]:
from airq.config import load_config
from airq.env import mount_drive_if_wanted
from airq.pipelines.colab import env_summary

mount_drive_if_wanted(USE_DRIVE)
cfg = load_config()
env_summary(cfg)


In [ ]:
import pandas as pd
from airq.env import is_fixture, resolve_paths
from airq.pipelines.data import run

paths = resolve_paths(cfg["paths"])
if is_fixture():
    from airq.data.fixture import make_fixture_raw
    make_fixture_raw(paths["raw"])
else:
    from airq.data.download import download_beijing
    info = download_beijing(cfg["data"]["beijing_url"], paths["raw"] / "beijing",
                            expected_sha256=cfg["data"]["beijing_sha256"])
    print("SHA-256 verified:", info["sha256"][:16], "...")
result = run(cfg, quick=QUICK)
result


In [ ]:
pd.read_csv(paths["reports"] / "tables" / "dataset_summary.csv")


In [ ]:
# --- export: reports/ and models/ -> airq_artifacts_<date>.zip --------------------------
from airq.pipelines.colab import export_artifacts

zip_path = export_artifacts(cfg)
print("Wrote", zip_path.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
# locally:  python -m scripts.import_artifacts <zip>   (also runs the manifest check)
